In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp04/ex_7.py ──
"""
Shared infrastructure for MLFP04 Exercise 7 — Recommender Systems.

Scenario: a fictional Singapore electronics marketplace.
  - 300 users x 120 SKUs, explicit 1-5 star ratings
  - Ratings follow a known generative model so every technique can be
    checked against the truth:
        rating = global mean + user bias + item bias
                 + (user taste vector . item trait vector) + noise
    with N_LATENT_TRUE = 5 hidden taste dimensions.
  - 30% of (user, item) pairs are observed
  - 30% of observed ratings are held out for evaluation (~10 per user, so
    precision@5 and MAP actually have to RANK items)
  - N_COLD_ITEMS brand-new SKUs have ALL their ratings in the holdout set:
    they are cold-start items that no collaborative method has seen
  - A validation split (VAL_FRAC of training ratings) is carved out for
    tuning hybrid blend weights — the holdout set is never used for tuning
  - Item content features = a noisy linear view of each SKU's hidden traits
    and quality (think price tier, category, spec sheet)

Contains: synthetic rating matrix generation, baseline predictors, evaluation
metrics (RMSE, precision@k, MAP), shared constants, and output-dir setup.

Technique-specific code (content-based profile, user/item CF, ALS, hybrid
blending) lives in the per-technique files.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex7_recommenders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS — synthetic SG e-commerce dataset
# ════════════════════════════════════════════════════════════════════════

N_USERS = 300
N_ITEMS = 120
N_LATENT_TRUE = 5
N_ITEM_FEATURES = 8
SPARSITY = 0.30  # fraction of (user, item) pairs that carry a rating
HOLDOUT_FRAC = 0.30  # fraction of observed ratings held out for evaluation
VAL_FRAC = 0.20  # fraction of TRAINING ratings reserved for blend tuning
N_COLD_ITEMS = 10  # brand-new SKUs whose every rating is in the holdout
RNG_SEED = 42

RATING_MIN = 1.0
RATING_MAX = 5.0
RELEVANCE_THRESHOLD = 3.5  # ratings >= 3.5 are "relevant" for ranking metrics

# Generative-model scales (documented so students can reason about them)
_GLOBAL_MEAN = 3.2
_USER_BIAS_SD = 0.35
_ITEM_BIAS_SD = 0.45
_TASTE_SCALE = 0.9
_NOISE_SD = 0.35
_FEATURE_NOISE_SD = 0.5


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC RATING MATRIX — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def build_rating_dataset(
    seed: int = RNG_SEED,
) -> dict:
    """Generate the shared synthetic e-commerce rating matrix.

    Returns a dict with:
      R_observed       — (N_USERS, N_ITEMS) with NaN where not rated
      R_train          — R_observed with holdout entries set to NaN
      mask             — boolean observed mask
      train_mask       — boolean training-only mask (holdout removed)
      holdout_mask     — boolean holdout (test) mask
      fit_mask         — train_mask minus the validation ratings
      val_mask         — validation ratings (subset of train_mask), used ONLY
                         for tuning blend weights in 05_hybrid_evaluation.py
      cold_items       — boolean (N_ITEMS,) — SKUs with zero training ratings
      U_true, V_true   — ground-truth latent taste / trait factors
      item_features    — (N_ITEMS, N_ITEM_FEATURES) content-feature matrix
      user_ids, item_ids — stable string IDs
      ratings_df       — long-format polars DataFrame of observed ratings
      rng              — the generator, for downstream reproducible draws
    """
    rng = np.random.default_rng(seed=seed)

    U_true = rng.normal(0, 1, size=(N_USERS, N_LATENT_TRUE))
    V_true = rng.normal(0, 1, size=(N_ITEMS, N_LATENT_TRUE))
    user_bias = rng.normal(0, _USER_BIAS_SD, size=N_USERS)
    item_bias = rng.normal(0, _ITEM_BIAS_SD, size=N_ITEMS)

    taste = U_true @ V_true.T / np.sqrt(N_LATENT_TRUE)  # ~N(0, 1) per pair
    R_full = (
        _GLOBAL_MEAN
        + user_bias[:, None]
        + item_bias[None, :]
        + _TASTE_SCALE * taste
        + rng.normal(0, _NOISE_SD, size=(N_USERS, N_ITEMS))
    )
    R_full = np.clip(R_full, RATING_MIN, RATING_MAX)

    mask = rng.random(size=(N_USERS, N_ITEMS)) < SPARSITY
    R_observed = np.where(mask, R_full, np.nan)

    cold_items = np.zeros(N_ITEMS, dtype=bool)
    cold_items[rng.choice(N_ITEMS, size=N_COLD_ITEMS, replace=False)] = True

    holdout_mask = mask & (rng.random(size=(N_USERS, N_ITEMS)) < HOLDOUT_FRAC)
    holdout_mask |= mask & cold_items[None, :]  # cold SKUs: every rating is test
    train_mask = mask & ~holdout_mask
    R_train = np.where(train_mask, R_observed, np.nan)

    val_mask = train_mask & (rng.random(size=(N_USERS, N_ITEMS)) < VAL_FRAC)
    fit_mask = train_mask & ~val_mask

    # Content features: a noisy linear view of each item's hidden traits
    # AND its quality (item bias). Real catalogues behave the same way —
    # spec sheets and price tiers correlate with, but do not equal, what
    # drives ratings.
    hidden = np.hstack(
        [V_true / np.sqrt(N_LATENT_TRUE), (item_bias / _ITEM_BIAS_SD)[:, None]]
    )
    W_feat = rng.normal(0, 1, size=(N_LATENT_TRUE + 1, N_ITEM_FEATURES))
    item_features = hidden @ W_feat / np.sqrt(N_LATENT_TRUE + 1) + rng.normal(
        0, _FEATURE_NOISE_SD, size=(N_ITEMS, N_ITEM_FEATURES)
    )

    user_ids = [f"sg_user_{i:03d}" for i in range(N_USERS)]
    item_ids = [f"sku_{j:03d}" for j in range(N_ITEMS)]

    rows = []
    for i in range(N_USERS):
        for j in range(N_ITEMS):
            if mask[i, j]:
                rows.append(
                    {
                        "user_id": user_ids[i],
                        "item_id": item_ids[j],
                        "rating": round(float(R_observed[i, j]), 1),
                        "in_holdout": bool(holdout_mask[i, j]),
                    }
                )
    ratings_df = pl.DataFrame(rows)

    return {
        "R_observed": R_observed,
        "R_train": R_train,
        "mask": mask,
        "train_mask": train_mask,
        "holdout_mask": holdout_mask,
        "fit_mask": fit_mask,
        "val_mask": val_mask,
        "cold_items": cold_items,
        "U_true": U_true,
        "V_true": V_true,
        "item_features": item_features,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "ratings_df": ratings_df,
        "rng": rng,
    }


# ════════════════════════════════════════════════════════════════════════
# BASELINES — every recommender must beat these to be worth deploying
# ════════════════════════════════════════════════════════════════════════


def baseline_predictions(
    R: np.ndarray, obs_mask: np.ndarray, seed: int = RNG_SEED
) -> dict[str, np.ndarray]:
    """Return three no-skill reference predictors fitted on ``R[obs_mask]``.

    - "Global mean":  every pair gets the mean training rating (RMSE floor)
    - "Item mean":    each SKU's mean training rating (popularity), falling
                      back to the global mean for SKUs with no training data
    - "Random":       uniform scores in [1, 5] — the chance line for P@k/MAP
    """
    global_mean = float(R[obs_mask].mean())
    n_users, n_items = R.shape
    counts = obs_mask.sum(axis=0)
    sums = np.where(obs_mask, R, 0.0).sum(axis=0)
    item_mean = np.where(counts > 0, sums / np.maximum(counts, 1), global_mean)
    rng = np.random.default_rng(seed)
    return {
        "Global mean": np.full((n_users, n_items), global_mean),
        "Item mean": np.tile(item_mean, (n_users, 1)),
        "Random": rng.uniform(RATING_MIN, RATING_MAX, size=(n_users, n_items)),
    }


# ════════════════════════════════════════════════════════════════════════
# EVALUATION METRICS — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def holdout_rmse(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
) -> tuple[float, float]:
    """Return (RMSE, coverage) on the holdout mask.

    Coverage = fraction of holdout pairs where the method produced a
    non-NaN prediction. Methods that can't predict cold pairs have
    coverage < 1.0; their RMSE is computed only on the pairs they cover.
    """
    n_holdout = int(holdout_mask.sum())
    covered = holdout_mask & ~np.isnan(predictions)
    if not covered.any():
        return float("inf"), 0.0
    errors = (R_true[covered] - predictions[covered]) ** 2
    rmse = float(np.sqrt(np.mean(errors)))
    coverage = float(covered.sum()) / max(n_holdout, 1)
    return rmse, coverage


def precision_at_k(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    k: int = 5,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Precision@k averaged across users.

    For each user, rank holdout items by predicted score, take the top-k,
    and compute what fraction of those have true rating >= threshold.
    Items a method cannot score (NaN) are never recommended.
    """
    precisions = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])
        top = [j for j, _ in scored[:k]]
        if not top:
            precisions.append(0.0)
            continue
        precisions.append(len(set(top) & relevant) / len(top))
    return float(np.mean(precisions)) if precisions else 0.0


def mean_average_precision(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Mean Average Precision across users on the holdout set.

    A relevant item the method cannot score counts as never retrieved, so
    low coverage lowers MAP.
    """
    aps = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])

        hits = 0
        sum_precision = 0.0
        for rank, (j, _) in enumerate(scored, 1):
            if j in relevant:
                hits += 1
                sum_precision += hits / rank
        aps.append(sum_precision / len(relevant))
    return float(np.mean(aps)) if aps else 0.0


def evaluate_method(
    preds: np.ndarray, R_true: np.ndarray, holdout_mask: np.ndarray
) -> dict:
    """Return {"RMSE", "Coverage", "P@5", "MAP"} for one prediction matrix."""
    rmse, cov = holdout_rmse(preds, R_true, holdout_mask)
    return {
        "RMSE": rmse,
        "Coverage": cov,
        "P@5": precision_at_k(preds, R_true, holdout_mask, k=5),
        "MAP": mean_average_precision(preds, R_true, holdout_mask),
    }


def print_method_scores(
    name: str, preds: np.ndarray, R_true: np.ndarray, holdout_mask: np.ndarray
) -> dict:
    """Print a single-line scorecard and return the metrics dict."""
    m = evaluate_method(preds, R_true, holdout_mask)
    print(
        f"  {name:<22} RMSE={m['RMSE']:6.4f}  coverage={m['Coverage']:6.1%}  "
        f"P@5={m['P@5']:6.4f}  MAP={m['MAP']:6.4f}"
    )
    return m


def print_baselines(
    R_train: np.ndarray,
    train_mask: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
) -> dict[str, dict]:
    """Score the three no-skill baselines on the holdout and print them."""
    print("  Baselines (a recommender must beat these):")
    return {
        name: print_method_scores(name, preds, R_true, holdout_mask)
        for name, preds in baseline_predictions(R_train, train_mask).items()
    }


def print_warm_comparison(
    name: str,
    preds: np.ndarray,
    R_train: np.ndarray,
    train_mask: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    cold_items: np.ndarray,
) -> tuple[float, float]:
    """Compare a CF-style method with the item-mean baseline on WARM SKUs.

    Collaborative methods cannot score cold-start SKUs, and MAP counts an
    unscored relevant item as a miss — so on the full holdout they can lose
    to a baseline that covers everything. Restricting to warm SKUs isolates
    ranking skill from coverage. Returns (method MAP, item-mean MAP).
    """
    warm = holdout_mask & ~cold_items[None, :]
    method_map = mean_average_precision(preds, R_true, warm)
    item_mean = baseline_predictions(R_train, train_mask)["Item mean"]
    base_map = mean_average_precision(item_mean, R_true, warm)
    verdict = "beats" if method_map > base_map else "does NOT beat"
    print(
        f"  Warm SKUs only: {name} MAP={method_map:.4f} vs item-mean "
        f"MAP={base_map:.4f} -> {name} {verdict} the popularity baseline "
        "on items it can see."
    )
    return method_map, base_map


# ════════════════════════════════════════════════════════════════════════
# VISUAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def save_html(fig, filename: str) -> Path:
    """Write a plotly fig into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    print(f"  saved: {path}")
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 7.5: Hybrid Recommender + Full Evaluation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Blend four recommenders into a hybrid with weights tuned on a
#     validation split (never on the test holdout)
#   - Compare every method on RMSE, coverage, precision@k, and MAP —
#     against no-skill baselines
#   - Understand why ranking metrics matter more than RMSE in production
#   - Explain implicit vs explicit feedback and when each applies
#   - See why Netflix Prize teams won by blending many models
#
# PREREQUISITES: Exercises 7.1, 7.2, 7.3, 7.4
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — when (and why) blends beat a single model
#   2. Build — re-run all four base recommenders and the hybrid blender
#   3. Train — tune the blend weights on the validation split
#   4. Visualise — side-by-side method comparison (RMSE + MAP)
#   5. Apply — Singapore news aggregator front-page personalisation
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from kailash_ml import ModelVisualizer
from scipy.optimize import nnls


K_LATENT = N_LATENT_TRUE
LAMBDA_REG = 5.0
N_ITERATIONS = 30
K_NEIGHBOURS = 20



## THEORY — When hybrids beat any single method


In [ ]:
# Every recommender in this exercise has a failure mode:
#   - Content-based: limited by feature quality, filter bubble
#   - User-CF: O(N^2) compute, cold-start users AND cold-start items
#   - Item-CF: niche-item sparsity, cold-start items
#   - ALS MF: popularity bias, cold-start everything
#
# A hybrid combines their predictions. It helps when the components make
# DIFFERENT mistakes — e.g. CF methods cannot score a brand-new SKU at all,
# while content-based can. If one component is better than the others
# everywhere, a well-tuned blend simply puts (almost) all its weight on
# that component — and a badly-tuned blend makes things worse.
#
# Blend weights are model parameters, so they must be tuned on data the
# final evaluation never sees. We split the training ratings into a FIT
# part (base models learn from it) and a VALIDATION part (blend weights
# are chosen on it). The test holdout is used exactly once, at the end.
#
# Here the weights come from non-negative least squares (NNLS): find
# w >= 0 minimising sum_val (r - sum_m w_m * pred_m)^2. That is a tiny
# "stacking" model. The Netflix Prize teams used far richer stacked
# blends: the 2007 Progress Prize entry combined 107 models, and the 2009
# winning entry blended several hundred.
#
# RMSE vs ranking metrics:
#   RMSE measures rating prediction accuracy. Precision@k and MAP measure
#   whether the TOP recommendations are actually good. For production
#   systems, ranking matters more — users never see predictions 10-50.



## TASK 2 — BUILD: rerun the four base models and the blender


Missing component predictions are filled with ``fill_value`` (the
    global training mean) for the fit only.


For each (user, item): average the available component predictions
    with the tuned weights (renormalised over the available ones). If no
    weighted component can score the pair — e.g. a cold-start SKU that
    only content-based can see — use ``fallback`` instead.


In [ ]:
# To keep this file standalone we inline the four algorithms from
# 01-04 (same maths, compact form) so it runs top-to-bottom.


def content_based_predict(R, item_feats, obs_mask):
    n_users, n_items = R.shape
    preds = np.full((n_users, n_items), np.nan)
    feats_c = item_feats - item_feats.mean(axis=0)
    for u in range(n_users):
        rated = np.where(obs_mask[u])[0]
        if len(rated) == 0:
            continue
        mean_u = float(R[u, rated].mean())
        std_u = float(R[u, rated].std()) + 1e-9
        profile = ((R[u, rated] - mean_u)[:, None] * feats_c[rated]).sum(axis=0)
        pn = np.linalg.norm(profile)
        if pn < 1e-10:
            continue
        profile /= pn
        for j in range(n_items):
            fn = np.linalg.norm(feats_c[j])
            if fn < 1e-10:
                continue
            preds[u, j] = mean_u + 2.0 * std_u * (profile @ feats_c[j] / fn)
    return np.clip(preds, 1.0, 5.0)


def user_cf_predict(R, obs_mask, k=K_NEIGHBOURS):
    n_users, n_items = R.shape
    user_means = np.array(
        [
            float(np.nanmean(R[u, obs_mask[u]])) if obs_mask[u].any() else 0.0
            for u in range(n_users)
        ]
    )
    Rc = R.copy()
    for u in range(n_users):
        Rc[u, obs_mask[u]] -= user_means[u]
    Rc[~obs_mask] = 0.0
    sim = np.zeros((n_users, n_users))
    for u in range(n_users):
        for v in range(u, n_users):
            both = obs_mask[u] & obs_mask[v]
            if not both.any():
                continue
            ru, rv = Rc[u, both], Rc[v, both]
            d = np.linalg.norm(ru) * np.linalg.norm(rv)
            if d < 1e-10:
                continue
            s = float(ru @ rv / d)
            sim[u, v] = s
            sim[v, u] = s
    preds = np.full((n_users, n_items), np.nan)
    for u in range(n_users):
        s = sim[u].copy()
        s[u] = -np.inf
        top = np.argsort(s)[-k:]
        top = top[s[top] > 0]
        if len(top) == 0:
            continue
        for j in range(n_items):
            rated_n = top[obs_mask[top, j]]
            if len(rated_n) == 0:
                continue
            w = sim[u, rated_n]
            denom = np.abs(w).sum()
            if denom < 1e-10:
                continue
            preds[u, j] = (
                user_means[u] + (w @ (R[rated_n, j] - user_means[rated_n])) / denom
            )
    return np.clip(preds, 1.0, 5.0)


def item_cf_predict(R, obs_mask, k=K_NEIGHBOURS):
    n_users, n_items = R.shape
    item_means = np.array(
        [
            float(np.nanmean(R[obs_mask[:, j], j])) if obs_mask[:, j].any() else 0.0
            for j in range(n_items)
        ]
    )
    Rc = R.copy()
    for j in range(n_items):
        Rc[obs_mask[:, j], j] -= item_means[j]
    Rc[~obs_mask] = 0.0
    sim = np.zeros((n_items, n_items))
    for i in range(n_items):
        for j in range(i, n_items):
            both = obs_mask[:, i] & obs_mask[:, j]
            if not both.any():
                continue
            ri, rj = Rc[both, i], Rc[both, j]
            d = np.linalg.norm(ri) * np.linalg.norm(rj)
            if d < 1e-10:
                continue
            s = float(ri @ rj / d)
            sim[i, j] = s
            sim[j, i] = s
    preds = np.full((n_users, n_items), np.nan)
    for u in range(n_users):
        rated = np.where(obs_mask[u])[0]
        if len(rated) == 0:
            continue
        for j in range(n_items):
            sims = sim[j, rated]
            top_idx = np.argsort(sims)[-k:] if len(sims) > k else np.arange(len(sims))
            pos = top_idx[sims[top_idx] > 0]
            if len(pos) == 0:
                continue
            w = sims[pos]
            denom = np.abs(w).sum()
            if denom < 1e-10:
                continue
            dev = R[u, rated[pos]] - item_means[rated[pos]]
            preds[u, j] = item_means[j] + (w @ dev) / denom
    return np.clip(preds, 1.0, 5.0)


def als_predict(R, obs_mask, k, lam, n_iter, rng):
    n_users, n_items = R.shape
    mu = float(R[obs_mask].mean())
    U = rng.normal(0, 0.1, size=(n_users, k))
    V = rng.normal(0, 0.1, size=(n_items, k))
    b_user = np.zeros(n_users)
    b_item = np.zeros(n_items)
    R_safe = np.nan_to_num(R, nan=0.0)
    penalty = lam * np.eye(k + 1)
    for _ in range(n_iter):
        for u in range(n_users):
            rated = np.where(obs_mask[u])[0]
            if len(rated) == 0:
                continue
            X = np.hstack([V[rated], np.ones((len(rated), 1))])
            y = R_safe[u, rated] - mu - b_item[rated]
            sol = np.linalg.solve(X.T @ X + penalty, X.T @ y)
            U[u], b_user[u] = sol[:k], sol[k]
        for j in range(n_items):
            raters = np.where(obs_mask[:, j])[0]
            if len(raters) == 0:
                continue
            X = np.hstack([U[raters], np.ones((len(raters), 1))])
            y = R_safe[raters, j] - mu - b_user[raters]
            sol = np.linalg.solve(X.T @ X + penalty, X.T @ y)
            V[j], b_item[j] = sol[:k], sol[k]
    preds = np.clip(mu + b_user[:, None] + b_item[None, :] + U @ V.T, 1.0, 5.0)
    preds[:, obs_mask.sum(axis=0) == 0] = np.nan
    return preds


def fit_blend_weights(
    all_preds: dict, R_true: np.ndarray, val_mask: np.ndarray, fill_value: float
) -> dict:
    names = list(all_preds)
    X_val = np.column_stack(
        [np.nan_to_num(all_preds[n][val_mask], nan=fill_value) for n in names]
    )
    y_val = R_true[val_mask]
    w, _ = nnls(X_val, y_val)
    return dict(zip(names, w))


def blend_hybrid(all_preds: dict, weights: dict, fallback: np.ndarray) -> np.ndarray:
    names = list(all_preds)
    stack = np.stack([all_preds[n] for n in names])
    w = np.array([weights[n] for n in names])[:, None, None]
    available = ~np.isnan(stack)
    w_avail = w * available
    denom = w_avail.sum(axis=0)
    blended = (w_avail * np.nan_to_num(stack)).sum(axis=0) / np.where(
        denom > 0, denom, 1.0
    )
    return np.where(denom > 0, blended, fallback)



## TASK 3 — FIT base models, TUNE blend on validation, EVALUATE on holdout


In [ ]:
print("\n" + "=" * 70)
print("  Hybrid Recommender + Full Evaluation")
print("=" * 70)

data = build_rating_dataset()
R_observed = data["R_observed"]
fit_mask = data["fit_mask"]
val_mask = data["val_mask"]
holdout_mask = data["holdout_mask"]
item_features = data["item_features"]
cold_items = data["cold_items"]
R_fit = np.where(fit_mask, R_observed, np.nan)
global_mean = float(R_fit[fit_mask].mean())

print(
    f"\nSplit sizes: fit={int(fit_mask.sum())}  validation={int(val_mask.sum())}  "
    f"test holdout={int(holdout_mask.sum())} ratings "
    f"({int(cold_items.sum())} cold-start SKUs appear only in the holdout)"
)

print("\nRunning base recommenders on the FIT split...")
cb = content_based_predict(R_fit, item_features, fit_mask)
ucf = user_cf_predict(R_fit, fit_mask)
icf = item_cf_predict(R_fit, fit_mask)
als = als_predict(R_fit, fit_mask, K_LATENT, LAMBDA_REG, N_ITERATIONS, data["rng"])

all_predictions = {
    "Content-Based": cb,
    "User-CF": ucf,
    "Item-CF": icf,
    "ALS MF": als,
}

blend_weights = fit_blend_weights(all_predictions, R_observed, val_mask, global_mean)
print("\nBlend weights (NNLS on the validation split):")
for name, w in blend_weights.items():
    print(f"  {name:<18} {w:.3f}")
hybrid_preds = blend_hybrid(all_predictions, blend_weights, fallback=cb)

eval_results: dict = {}
for name, preds in baseline_predictions(R_fit, fit_mask).items():
    eval_results[f"[baseline] {name}"] = evaluate_method(
        preds, R_observed, holdout_mask
    )
for name, preds in all_predictions.items():
    eval_results[name] = evaluate_method(preds, R_observed, holdout_mask)
eval_results["Hybrid"] = evaluate_method(hybrid_preds, R_observed, holdout_mask)

print(f"\n{'Method':<24} {'RMSE':>8} {'Coverage':>10} {'P@5':>8} {'MAP':>8}")
print("─" * 62)
for name, r in eval_results.items():
    print(
        f"{name:<24} {r['RMSE']:>8.4f} {r['Coverage']:>9.1%} "
        f"{r['P@5']:>8.4f} {r['MAP']:>8.4f}"
    )

base_methods = list(all_predictions)
best_single = max(base_methods, key=lambda n: eval_results[n]["MAP"])
best_single_map = eval_results[best_single]["MAP"]
hybrid_map = eval_results["Hybrid"]["MAP"]
lift = hybrid_map - best_single_map
chance_map = eval_results["[baseline] Random"]["MAP"]
print(f"\nBest single method by MAP: {best_single} ({best_single_map:.4f})")
print(f"Hybrid MAP:                {hybrid_map:.4f}  (lift {lift:+.4f})")
print(f"Random-ranking MAP:        {chance_map:.4f}")

# Where does the hybrid's lift come from? Score warm SKUs on their own.
warm_holdout = holdout_mask & ~cold_items[None, :]
warm_map = {
    name: evaluate_method(preds, R_observed, warm_holdout)["MAP"]
    for name, preds in all_predictions.items()
}
best_warm = max(warm_map, key=warm_map.get)
hybrid_warm_map = evaluate_method(hybrid_preds, R_observed, warm_holdout)["MAP"]
warm_gain = hybrid_warm_map - warm_map[best_warm]
top_component = max(blend_weights, key=blend_weights.get)
print(
    f"Warm SKUs only: hybrid MAP {hybrid_warm_map:.4f} vs best component "
    f"{best_warm} {warm_map[best_warm]:.4f} (gain {warm_gain:+.4f}); "
    f"largest blend weight: {top_component} ({blend_weights[top_component]:.3f})"
)
if lift > 0.005 and abs(warm_gain) <= 0.01:
    print(
        "  -> The hybrid beats every single method, but on warm SKUs it only "
        f"matches {best_warm}: the gain comes from COVERAGE — the hybrid "
        "hands the cold-start SKUs that CF/MF cannot score to content-based."
    )
elif lift > 0.005:
    print(
        "  -> The hybrid beats every single method, and it also improves on "
        "the best component for warm SKUs — the components' errors are "
        "complementary there too."
    )
elif lift > -0.005:
    print(
        "  -> The hybrid ties the best single method: blending adds little "
        "on this data."
    )
else:
    print(
        "  -> The hybrid is WORSE than the best single method on this run — "
        "blending diluted the winner. Check the weights and the split sizes."
    )



### Checkpoint


In [ ]:
assert len(eval_results) == 8, "Should have 3 baselines + 4 base methods + hybrid"
assert eval_results["Hybrid"]["Coverage"] == 1.0, "Hybrid must score every pair"
assert all(w >= 0 for w in blend_weights.values()), "NNLS weights are non-negative"
assert 0.0 <= eval_results["Hybrid"]["P@5"] <= 1.0, "Precision@k must be in [0, 1]"
assert 0.0 <= hybrid_map <= 1.0, "MAP must be in [0, 1]"
assert best_single_map > chance_map, "Best recommender must beat random ranking"
print("\n[ok] Checkpoint passed — all methods evaluated on RMSE, P@5, MAP\n")



## TASK 4 — VISUALISE: side-by-side comparison


In [ ]:
# A metric comparison chart makes trade-offs obvious: a method with low
# RMSE on the pairs it covers can still have poor MAP if it cannot score
# the cold-start SKUs at all.

viz = ModelVisualizer()
comparison_metrics = {
    name: {"RMSE": r["RMSE"], "MAP": r["MAP"], "Coverage": r["Coverage"]}
    for name, r in eval_results.items()
    if name != "[baseline] Random"  # random-score RMSE dwarfs the axis
}
fig_cmp = viz.metric_comparison(comparison_metrics)
fig_cmp.update_layout(title="Recommender Method Comparison (RMSE, MAP, coverage)")
save_html(fig_cmp, "05_method_comparison.html")



## IMPLICIT vs EXPLICIT FEEDBACK — conceptual summary


Explicit feedback: users provide ratings (1-5 stars).
  + clear signal
  - sparse (most users rate few items)
  - absence of rating is ambiguous

Implicit feedback: clicks, views, purchases, time spent.
  + abundant
  - noisy (click != like)
  - no NEGATIVE signal (no click could mean "never seen")

ALS for implicit data (Hu, Koren & Volinsky, 2008):
  - Treat ALL pairs as observed
  - p[u,i] = 1 if user interacted, 0 otherwise
  - c[u,i] = 1 + alpha * count  (confidence)
  - Loss: sum_all c(u,i) * (p(u,i) - U[u] V[i])^2 + lambda * (||U||^2 + ||V||^2)

Most large consumer platforms have far more implicit than explicit
feedback, so implicit-feedback MF and its neural successors are common.


In [ ]:
print(
)



## TASK 5 — APPLY: Singapore News Aggregator Front-Page Personalisation


In [ ]:
# SCENARIO: A Singapore news aggregator runs personalised front pages for
# ~3M daily active users. The front page has 8 slots. Every refresh must
# rank hundreds of articles by predicted relevance in <50ms.
#
# Why a HYBRID is the right tool:
#   - BREAKING NEWS = cold-start items (0 reads). Content-based kicks in
#     using tags, section, author, entity mentions — exactly the role it
#     played for the cold-start SKUs above
#   - REGULAR CONTENT = CF / MF win via co-read patterns
#   - A blend tuned on held-out validation data picks how much to trust
#     each lever; your measured weights above show what that looks like
#
# BUSINESS IMPACT (illustrative assumptions, not measured figures): on an
# aggregator with S$35M annual ad revenue, if personalisation lifted
# clicks-per-session by 10% and ad revenue scaled with clicks, that would
# be ~S$3.5M/year — against perhaps S$400K/year of ML infrastructure and
# engineering. Only an online A/B test can confirm the lift; offline MAP
# tells you which candidate to put into that test.
#
# LIMITATIONS of simple global blending:
#   - Weights are global; a per-user or per-segment gate (learn which
#     method to trust for WHICH user or item type) can do better
#   - A richer stacking model (e.g. gradient boosting on component scores
#     plus user/item counts) can capture interactions NNLS cannot
#   - At scale, a 4-method blend costs 4x inference — teams often distil
#     the ensemble into a single neural model (Exercise 8 builds the
#     neural-network foundations)



## REFLECTION


[x] Evaluated four recommenders + three baselines on RMSE, coverage, P@5, MAP
  [x] Tuned blend weights on a validation split, never on the test holdout
  [x] Measured the hybrid lift vs best single method ({lift:+.4f} MAP)
  [x] Explained implicit vs explicit feedback and ALS-implicit
  [x] Sized an (illustrative) SEA news scenario for hybrid recommenders

  KEY INSIGHT: A hybrid only helps when its components make different
  mistakes. Here the clearest difference is coverage — collaborative
  methods are blind to brand-new items, content-based is not.

  Exercise 7 complete — you now understand the full recommender stack.

  Next: Exercise 8 — neural networks generalise matrix factorisation by
  adding non-linear activations. The hidden layer IS an embedding,
  learned by the same principle.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

